# From data to a simple analysis: asking an LLM

In Lecture 7 we searched 200 UN speeches for the word *energy*. In Lecture 8 we plotted the hits and downloaded text from energy websites.

Both times, **we** wrote the rule: "is this word equal to `energy`?", "take the `<p>` tags". Today we replace the rule with a **question to a language model (LLM)**, and keep everything else the same: the same files, the same loops, the same tables.

Our questions:

1. Does an LLM find energy paragraphs that our keyword search missed?
2. *How* do countries talk about energy: as climate, security, economy or development?
3. Can an LLM read the pages we scraped in Lecture 8, and can we trust what it says?

Run cells with **Shift + Enter**. Complete the cells with `# ...` comments.

## 0. Open our tools

All of these are known from earlier lectures. `getpass` is new: it lets you type a password without it appearing in the notebook.

In [ ]:
import pandas as pd
import requests
import time
from getpass import getpass
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Connect to GPUStack

GPUStack is the University of Bern's own LLM server. The text we send **stays at the university**; it is not sent to OpenAI or Google. You still should never send personal or confidential data to a model you have not checked.

To use it you need a personal **API key**: a long password that identifies you to the server.

1. Go to <https://gpustack.unibe.ch> and log in with your Campus Account.
2. Open **API Keys** and click **Create API Key**. Give it a name, e.g. `wti`.
3. Copy the key. **It is shown only once.** Keep it somewhere safe.

Run the cell below and paste your key into the box that appears. Never write your key directly into a notebook: notebooks get shared and uploaded.

In [ ]:
api_key = getpass("Paste your GPUStack API key: ")
base_url = "https://gpustack.unibe.ch/v1"

### Which models are available?

Asking the server for its list of models is an ordinary web request, exactly like in Lecture 8. The only difference is the `headers`: that is where we show our key.

In [ ]:
response = requests.get(base_url + "/models", headers={"Authorization": "Bearer " + api_key}, timeout=20)
print(response.status_code)

for m in response.json()["data"]:
    print(m["id"])

**Status 403** ("Access denied")? GPUStack only answers inside the university network. Connect to the **university VPN** (or eduroam on campus) and run the cell again.

**Status 401?** The key is wrong or was not copied completely. Run the key cell again.

We all use the same model today: `gpt-oss-120b`. It is fast, which matters when the whole class asks at the same time.

In [ ]:
model = "gpt-oss-120b"       # fast. The slower, more careful one: "qwen3.8-27b"

## 2. Ask one question

An LLM question is a `requests.post()`: we **send** something to the server (in Lecture 8 we only *got* pages). We send a small dictionary:

- `model`: which model should answer,
- `messages`: the conversation so far. Here just one message from us (the `"user"`),
- `temperature`: 0 means "always give your most likely answer". This makes results more repeatable.

In [ ]:
question = "In one sentence: what is the UN General Debate?"

data = {
    "model": model,
    "messages": [{"role": "user", "content": question}],
    "temperature": 0,
}
response = requests.post(base_url + "/chat/completions",
                         headers={"Authorization": "Bearer " + api_key},
                         json=data, timeout=120)
print(response.status_code)

The answer comes back as JSON: dictionaries inside lists inside dictionaries. Look at it once, then we dig out the text.

In [ ]:
response.json()

In [ ]:
answer = response.json()["choices"][0]["message"]["content"]
print(answer)

### Make it reusable

As in Lecture 8 (`scrape_page`), we put these steps into a function. `ask_llm(prompt)` takes a question and returns the answer as a string. We will use this one function for the rest of the lecture.

Two additions compared with the cell above:

- A **system** message: an instruction that applies to the whole conversation. Without it, the model formats its answers for a chat window, with `**bold**` text and bullet points.
- `.replace()`: the model sometimes uses special typographic spaces and hyphens (`\u202f`, `\u2011`). We replace them with normal ones. `.strip()` removes spaces and line breaks around the answer.

Always use `print(ask_llm(...))`. Without `print`, Jupyter shows the raw string, with `\n` instead of line breaks.

In [ ]:
def ask_llm(prompt):
    data = {
        "model": model,
        "messages": [{"role": "system", "content": "Answer in plain text. Do not use Markdown, bold text or bullet points."},
                     {"role": "user", "content": prompt}],
        "temperature": 0,
    }
    response = requests.post(base_url + "/chat/completions",
                             headers={"Authorization": "Bearer " + api_key},
                             json=data, timeout=120)
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]
    answer = answer.replace("\u202f", " ").replace("\u00a0", " ").replace("\u2011", "-")   # special spaces and hyphens
    return answer.strip()

In [ ]:
# Ask the model any short question with ask_llm() and print the answer
# ...

## 3. Keyword or LLM? One speech, paragraph by paragraph

We go back to the speeches from Lecture 7 and take one short speech: **France, 2016** (row 25). Each line of the text is one paragraph, so `split("\n")` gives us a list of paragraphs (Lecture 7, section 8).

In [ ]:
speeches = pd.read_csv("ungd_speeches.csv")
print(speeches.loc[25, "country_name"], speeches.loc[25, "year"])
text = speeches.loc[25, "text"]
paragraphs = text.split("\n")
print(len(paragraphs))
print(paragraphs[4])

**Predict:** this paragraph does not contain the word *energy*. Does it talk about energy?

### The prompt

A **prompt** is the text we send to the model. We build it from a fixed instruction plus one paragraph. Asking for *only yes or no* makes the answer easy to use in Python.

In [ ]:
instruction = ("Does the following paragraph from a UN speech talk about energy "
               "(for example electricity, oil, gas, fuels, renewables)? "
               "Answer only yes or no.\n\nParagraph: ")

prompt = instruction + paragraphs[4]
print(ask_llm(prompt))

### The loop: same pattern as Lecture 7

In Lecture 7, section 7, the loop was: **create** an empty list, **update** it with one dictionary per hit, **use** it as a table. Today's loop has the same three parts. For every paragraph we write down two answers:

- `keyword`: our Lecture 7 rule: is the word `energy` in it?
- `llm`: the model's answer.

**Predict:** will the two columns always agree? Where could they differ?

In [ ]:
rows = []                                   # create

for paragraph in paragraphs:
    keyword_found = "energy" in paragraph.lower()
    # ... ask the LLM: build the prompt and call ask_llm()
    # ... make a dictionary with "paragraph", "keyword" and "llm", and append it to rows

table = pd.DataFrame(rows)                  # use
table.head()

### Compare the two methods

`value_counts()` on **two** columns counts every combination.

In [ ]:
table[["keyword", "llm"]].value_counts()

Did the model always answer exactly `yes` or `no`? If you see other answers (`yes.`, `Yes, it does`), that is your first lesson: **an LLM does not always follow instructions.** Our code must check.

Now read the paragraphs where the methods **disagree**: the keyword says no, the model says yes.

In [ ]:
for index, row in table.iterrows():
    if row["keyword"] == False and row["llm"].startswith("yes"):
        print(row["paragraph"])
        print("---")

**Discuss:** Is the model right? Which of these paragraphs would our Lecture 7 keyword search never have found?

### A harder speech

Row 98 is **United Kingdom, 2019**. The speech is about technology and data, not energy.

**Predict:** will the model and the keyword agree here?

Change `25` to `98` in the cell that loads the speech, then run it and every cell down to here again (**Run → Run Selected Cell and All Below**, then stop at this point).

**Discuss:** Read the paragraphs the model marked *yes*. Are they about energy, or do they only *use energy words* ("data is the new oil")? A keyword counts words; a model guesses meaning, and it can guess wrong. Who decides what "talks about energy" means: you or the model?

## 4. *How* do countries talk about energy?

The keyword search told us **where** energy is mentioned. It cannot tell us **how**. We give the model the `context` column from `energy_hits.csv` (10 words before and after each hit) and ask it to choose one **frame**.

In [ ]:
hits = pd.read_csv("energy_hits.csv")
hits[["country_name", "year", "context"]].head()

In [ ]:
frames = ["climate", "security", "economy", "development", "other"]

instruction = ("Here is a short extract from a UN speech that mentions energy. "
               "In this extract, energy is mainly discussed as a question of: "
               "climate, security, economy, development, or other? "
               "Answer with one word from this list only.\n\nExtract: ")

print(hits.loc[0, "context"])
print(ask_llm(instruction + hits.loc[0, "context"]))

### Start small

247 rows means 247 questions, and the whole class shares one server. We first take **40 random rows**. `sample(40, random_state=1)` picks them; `random_state=1` makes everyone get the same 40.

In [ ]:
sample = hits.sample(40, random_state=1)
sample.shape

### Your loop

This is the outer loop of Lecture 7, section 7: `iterrows()` gives us one row at a time.

The model may still answer something that is not in our list. We **check** it: if the answer is not one of our `frames`, we write `"unclear"`. Then we keep the original answer too, so we can look at it later.

In [ ]:
labels = []                                       # create

for index, row in sample.iterrows():
    answer = ask_llm(instruction + row["context"]).lower().strip(".")   # "Climate." -> "climate"
    # ... if answer is in frames, frame = answer, otherwise frame = "unclear"
    # ... append a dictionary with country_name, year, context, frame and answer to labels
    print(index, frame)

labelled = pd.DataFrame(labels)                   # use
labelled.head()

### Plot it (Lecture 8)

In [ ]:
# Draw a countplot of the "frame" column, with the frames on the y axis
# ...

In Lecture 8 we saw a spike of energy mentions in **2022**, the year after Russia's invasion of Ukraine. Does the *frame* change too? 40 rows is very little, so only read this as a hint.

In [ ]:
sns.countplot(data=labelled, x="year", hue="frame")
plt.title("Frames per year (40 extracts)")
plt.show()

### Check the model yourself

The model gives an answer for every row, but that does not make the answer right. Read 5 extracts and decide **yourself** before you look at the model's label.

In [ ]:
for index, row in labelled.head(5).iterrows():
    print(row["country_name"], row["year"])
    print(row["context"])
    print()

In [ ]:
labelled[["context", "frame"]].head(5)

**Discuss:** In how many of the 5 did you agree with the model? Would two people in the room agree with each other? A label you cannot check is not a result.

*If the server is not busy:* replace `sample` with `hits` in the loop to label all 247 rows, then save the table with `labelled.to_csv("energy_frames.csv", index=False)`.

## 5. Read the scraped pages (Lecture 8)

In Lecture 8 we saved `scraping_results.csv`: one row per website, with the extracted text, the HTTP status, and the errors. A copy is in this folder.

There, a selector decided what counted as text. Here, we give the text to the model and ask it **questions about the page**.

In [ ]:
pages = pd.read_csv("scraping_results.csv")
pages[["Title", "http_status", "status", "character_count"]]

### One page first

Long pages cost time. We send at most the first 4,000 characters: `text[:4000]`, slicing from Lecture 2.

In [ ]:
page_instruction = ("Below is text extracted from a web page. In two sentences: "
                    "who publishes this page, and what does it say about energy? "
                    "If the text is mostly a cookie notice or a menu, say so.\n\nText: ")

text = pages.loc[2, "text"]
print(pages.loc[2, "Title"])
print(ask_llm(page_instruction + text[:4000]))

### All pages, but only those with text

Four pages were refused (403) in Lecture 8. Their `text` is empty. We skip them with the same `if` as always and write down why.

`pages.loc[index, "summary"] = ...` writes one value into one cell of the table. The column is created the first time.

In [ ]:
for index, row in pages.iterrows():
    if row["character_count"] > 0:
        # ... ask the model about row["text"][:4000] and store the answer in pages.loc[index, "summary"]
        print("done:", row["Title"])
    else:
        pages.loc[index, "summary"] = "no text: " + str(row["status"])

pages[["Title", "summary"]]

In [ ]:
for index, row in pages.iterrows():
    print(row["Title"])
    print(row["summary"])
    print("---")

**Check:** open two of these pages in your browser. Is the summary correct? What did the model say about the **Swiss Energy Program** page, whose text started with an Italian cookie notice?

### Why did we skip the failed pages?

Row 6 is the International Atomic Energy Agency. In Lecture 8 the site refused us (403), so we have **no text** from it. What if we send the model only the address instead?

**Predict:** what will it answer?

In [ ]:
print(pages.loc[6, "Title"], "|", pages.loc[6, "status"])
print(ask_llm(page_instruction + pages.loc[6, "Url"]))

Read the answer carefully. Did the model say it could not see the page? Or did it describe the page's text, which it has never seen? The model cannot open web pages; it only saw the URL. An invented answer looks exactly like a real one. That is why we only send what we downloaded, keep the failed rows visible, and read the results. The LLM changed *how* we extract information, not the rule from Lecture 8: **missing data is missing, and every result must be checked.**

In [ ]:
pages.to_csv("pages_with_summaries.csv", index=False)

## 6. Your turn

Choose one:

1. **New frames.** Change the `frames` list and the instruction in section 4, e.g. `["renewable", "fossil", "nuclear", "general"]`. Run the loop again. Does the plot tell a different story?
2. **Your own keyword.** If you saved `trade_hits.csv` (or another keyword) in Lecture 7, copy it here and label its `context` column. Which frames make sense for *trade*?
3. **A better page question.** In section 5, ask for something you could put in a table, e.g. *"Answer with one word: in which language is this text?"* Store it in a new column `language`.

4. **Another model.** Set `model = "qwen3.8-27b"`, run the cell that defines `model`, then run the section 4 loop again. Do the two models give the same frames? If they disagree, which one would you trust, and how would you decide?

For each one: read 5 results yourself before you trust the plot.

## What we did today

| Step | Code |
|---|---|
| Type a secret without saving it | `api_key = getpass("...")` |
| Ask the server (send data) | `requests.post(url, headers=..., json=data)` |
| Get the answer text | `response.json()["choices"][0]["message"]["content"]` |
| Reuse it | `def ask_llm(prompt): ... return ...` |
| Prompt = instruction + data | `ask_llm(instruction + paragraph)` |
| Same loop as Lecture 7 | create `[]` → `append({...})` → `pd.DataFrame(...)` |
| Check the model's answer | `if answer in frames: ... else: "unclear"` |
| A random subset | `hits.sample(40, random_state=1)` |
| Write one cell | `pages.loc[index, "summary"] = ...` |

**The rule that does not change:** a keyword, a selector, or an LLM: each one gives you a result for every row. Only reading the results tells you whether they are right.